# 03 - Train a DQN Model Online

This notebook shows the online training workflow in Mouse Core. Instead of loading a fixed dataset, it keeps live environments in the loop:

1. Step a `GroupEnv` to collect fresh transitions.
2. Append transitions into in-memory `Datastore` replay streams.
3. Sample those streams with `DataLoader`.
4. Train with `DqnObjective`.
5. Repeat rollout and training until the optimizer budget is reached.

The important Mouse Core idea is that online and offline training use the same row format, model interface, datastores, dataloader, and objective. Only the source of rows changes.


In [1]:
from collections import deque



import torch



import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1

from mouse_gym import EnvConfig, make_group_env

from mouse_core.data import DataLoader, Datastore

from mouse_core.models import Model, push_model_to_hub

from mouse_core.models.backbone import Qwen3Backbone

from mouse_core.models.embedding import StepEmbedder

from mouse_core.models.heads import DiscreteActionValueHead

from mouse_core.objectives import DqnObjective





MODEL_ID = "mouse-example-model-online"  # Hugging Face model repo for push_model_to_hub

MAX_ACTIONS = 4  # number of discrete actions predicted by the head

MAX_OBS_DISCRETE = 64  # vocabulary size for discrete observations

MAX_EPISODE_STEPS = 30  # environment-specific episode limit

EPISODES_PER_TASK = 20  # environment-specific task length

NUM_ENVS = 30  # number of environment streams in the GroupEnv



GRADIENT_STEPS = 20000  # total optimizer updates

SEQUENCE_LENGTH = 512  # replay sequence length sampled by DataLoader

BATCH_SIZE = 4  # sequences per optimizer step



STEPS_PER_ROLLOUT = 500  # new rows collected per env before training

GRADIENT_STEPS_PER_ROLLOUT = 200  # optimizer updates after each rollout



LEARNING_STARTS = 15_000  # replay rows collected before the first optimizer update

EXPLORATION_ENDS = 1_500_000  # env-step horizon for epsilon decay



# Rollout rows are appended to datastores and become replay samples after loader.refresh().





device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



## Environment

Online training uses the same `EnvConfig` and `make_group_env` pattern as data collection. The group environment steps all configured instances together and returns one output dictionary per instance.

For your own environment, keep the row fields consistent with the model and objective you plan to use. In this notebook the model consumes `action`, `observation`, `reward`, and `done`; the DQN objective also uses `done` to decide whether bootstrapping should continue across a boundary.


In [2]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_online_{i}",
        reset_seed=i,
        episodes_per_task=EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},  # forwarded to the environment at task reset
        kwargs={
            "width": 8,
            "height": 8,
            "max_episode_steps": MAX_EPISODE_STEPS,
            "map_seed": i,
            "slippery_success_rate": 1.0,  # environment-specific option
            "permute_obs": True,      # environment-specific option
            "permute_actions": True,  # environment-specific option
        },
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)

## Build The Model

This is the same model assembly pattern used by offline training: `StepEmbedder` for row fields, `Qwen3Backbone` for sequence processing, and `DiscreteActionValueHead` for action values.

Because the online policy calls the model during rollout, the same model object is used in two modes: `eval()` for action selection and `train()` for optimizer updates.


In [3]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1,
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1,
        },
    ],
    modality_fusion="sum",
    include_type_token=False,
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
model.backbone.model.compile()  # optional compile step for faster repeated forwards
print(model)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model(
  (encoder): StepEmbedder(
    (type_embedder): TypeEmbedder(
      (embed): ScaledEmbedding(64, 1024)
    )
    (modality_embedders): ModuleDict(
      (action): DiscreteEmbedder(
        (embed): ScaledEmbedding(4, 1024)
      )
      (observation): DiscreteEmbedder(
        (embed): ScaledEmbedding(64, 1024)
      )
      (reward): ScalarRFFEmbedder(
        (rff): RandomFourierFeatures()
      )
      (done): DiscreteEmbedder(
        (embed): ScaledEmbedding(5, 1024)
      )
    )
  )
  (backbone): Qwen3Backbone(
    (model): Qwen3Model(
      (embed_tokens): Embedding(1, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=10

## Replay Buffer And Policy Contexts

Each environment stream writes to one `Datastore`; together they act as the replay buffer. Each environment also keeps a bounded `contexts` deque containing the recent rows used for action selection.

The replay buffer is for training batches. The context deque is for policy inference during rollout. Keeping them separate makes it clear which history is sampled for learning and which history conditions the next action.


In [4]:
stores = [Datastore(name=name) for name in env.names]
contexts = [deque(maxlen=SEQUENCE_LENGTH) for _ in env.names]


## Rollout Phase

A rollout collects new rows from the live environments. The loop follows the same shape each step:

1. Decide which environments should act greedily and which should explore randomly.
2. For greedy environments, call `model(..., use_cache=True)` on pending context rows.
3. Convert predictions to actions with `model.get_action(...)`.
4. Step the `GroupEnv` with one input dictionary per environment.
5. Append each merged row to its datastore, context deque, and pending decode chunk.

`use_cache=True` returns a cache object that can be passed back into the next model call. This avoids recomputing the whole context on every action selection. The cache is local to a rollout; the next rollout rebuilds from the bounded context deques.


In [5]:
def epsilon_for_env_step(env_step: int) -> float:
    """Return the epsilon-greedy exploration rate for a global env step."""
    if EXPLORATION_ENDS <= 0:
        raise ValueError("EXPLORATION_ENDS must be positive.")
    frac = min(env_step / EXPLORATION_ENDS, 1.0)
    return 1.0 - frac


In [6]:
def rollout(
    model: Model,
    env,
    stores: list[Datastore],
    contexts: list[deque],
    env_steps: int,
    grad_steps: int,
) -> int:
    """Collect rollout rows and append them to replay datastores."""
    env.metrics.clear()
    model.eval()

    # The cache stores decoded context during this rollout. pending_rows holds
    # rows that still need to be fed through the cached model call.
    kv_cache = None
    pending_rows = [list(c) for c in contexts]

    for step in range(STEPS_PER_ROLLOUT):
        epsilon = epsilon_for_env_step(env_steps)
        greedy = [bool(c) and torch.rand(1).item() >= epsilon for c in contexts]
        if any(greedy):
            with torch.no_grad():
                predictions, _, kv_cache = model(pending_rows, cache=kv_cache, use_cache=True)
            actions = model.get_action(predictions, temperature=0.0, num_actions=MAX_ACTIONS)
            pending_rows = [[] for _ in contexts]
        random_inputs = env.sample_random_input()
        inputs = [
            {"action": actions[i].cpu().numpy()} if greedy[i] else random_inputs[i]
            for i in range(len(contexts))
        ]
        outputs = env.step(inputs)
        for i, out in enumerate(outputs):
            row = {**inputs[i], **out}
            row.pop("info", None)  # keep stored rows flat
            stores[i].append(row)
            contexts[i].append(row)
            pending_rows[i].append(row)
        env_steps += len(outputs)
        if (step + 1) % 100 == 0:
            task_scores = [r for env_tasks in env.metrics.task_cum_rewards for r in env_tasks]
            mean_task_score = sum(task_scores) / len(task_scores) if task_scores else float("nan")
            epsilon = epsilon_for_env_step(env_steps)
            print(
                f"  rollout step {step + 1}/{STEPS_PER_ROLLOUT} | env_step={env_steps} grad_step={grad_steps}"
                f" epsilon={epsilon:.3f} | {len(task_scores)} tasks"
                f" | mean task score {mean_task_score:.2f}/{EPISODES_PER_TASK}"
            )
            env.metrics.clear()  # reset environment metrics for the next progress window

    del kv_cache  # release rollout-only cache before training
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return env_steps


## Training Phase

After a rollout, training uses the same API as offline training. `loader.refresh()` tells the dataloader to rescan the growing datastores and clear prefetched batches, then the loop runs a fixed number of DQN updates.

`pack=True` lets the loader form full training sequences even while individual stores are still short. `weight_mode="per_step"` samples in proportion to the amount of data in each store.


In [7]:
loader = DataLoader(
    stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    weight_mode="per_step",
    pack=True,
    num_workers=0,
)
objective = DqnObjective(
    gamma_step=1.0,                 # discount for ordinary next-step bootstrapping
    gamma_episode_terminal=1.0,     # discount when an episode ends normally
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,        # discount when a task ends normally
    gamma_task_truncated=0.0,
    tau=0.0005,
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8,
)

In [8]:
def train(
    model: Model,
    optimizer: torch.optim.Optimizer,
    objective: DqnObjective,
    loader: DataLoader,
    env_steps: int,
    grad_steps: int,
    num_updates: int,
) -> tuple[torch.Tensor, dict[str, float]]:
    """Refresh replay and run ``num_updates`` optimizer updates."""
    model.train()
    loader.refresh()

    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for update in range(num_updates):
        batch = loader.next_batch()

        predictions, objective_data, _ = model(batch)
        loss, metrics = objective(objective_data, predictions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.polyak_update(action_value_tau=objective.tau)
        if (update + 1) % 100 == 0 or update + 1 == num_updates:
            print(
                f"env_step={env_steps} grad_step={grad_steps + update + 1}"
                f" | train loss={loss.item():.4f} q={metrics['q_values_mean']:.3f}"
            )

    assert loss is not None
    return loss, metrics


## Run Online Training

The main loop alternates rollout and training. `LEARNING_STARTS` delays optimizer updates until the replay stores contain enough rows. `GRADIENT_STEPS_PER_ROLLOUT` controls how much learning happens after each collection phase.


In [9]:
env_steps = 0
grad_steps = 0

while grad_steps < GRADIENT_STEPS:
    env_steps = rollout(model, env, stores, contexts, env_steps, grad_steps)

    if env_steps >= LEARNING_STARTS:
        num_updates = min(GRADIENT_STEPS_PER_ROLLOUT, GRADIENT_STEPS - grad_steps)
        train(model, optimizer, objective, loader, env_steps, grad_steps, num_updates)
        grad_steps += num_updates
        if device.type == "cuda":
            torch.cuda.empty_cache()  # release cached CUDA memory before the next rollout

loader.close()
env.close()
print(f"Online training finished ({grad_steps} optimizer steps, {env_steps} env steps).")


/home/user/Repos/mouse-core/.venv/lib/python3.13/site-packages/torch/nn/attention/flex_attention.py:2481: UserWarning: flex_attention called without torch.compile() - this will use an unfused implementation that materializes the full scores matrix instead of generating a fused kernel.

SOLUTION: Use torch.compile(flex_attention)(...)

If you want to debug your score_mod/mask_mod, you can set:
torch.nn.attention.flex_attention._FLEX_ATTENTION_DISABLE_COMPILE_DEBUG = True

This will allow you to use print statements or breakpoints. Note: This doesn't work with the backwards pass and may produce incorrect results.
  _warn_once(


  rollout step 100/500 | env_step=3000 grad_step=0 epsilon=0.998 | 3 tasks | mean task score 0.67/20
  rollout step 200/500 | env_step=6000 grad_step=0 epsilon=0.996 | 10 tasks | mean task score 2.50/20
  rollout step 300/500 | env_step=9000 grad_step=0 epsilon=0.994 | 12 tasks | mean task score 1.08/20
  rollout step 400/500 | env_step=12000 grad_step=0 epsilon=0.992 | 10 tasks | mean task score 3.90/20
  rollout step 500/500 | env_step=15000 grad_step=0 epsilon=0.990 | 16 tasks | mean task score 0.62/20
env_step=15000 grad_step=100 | train loss=0.0055 q=0.448
env_step=15000 grad_step=200 | train loss=0.0103 q=0.204
  rollout step 100/500 | env_step=18000 grad_step=200 epsilon=0.988 | 12 tasks | mean task score 1.67/20
  rollout step 200/500 | env_step=21000 grad_step=200 epsilon=0.986 | 17 tasks | mean task score 2.00/20
  rollout step 300/500 | env_step=24000 grad_step=200 epsilon=0.984 | 12 tasks | mean task score 2.00/20
  rollout step 400/500 | env_step=27000 grad_step=200 epsilo

[W711 02:04:42.159483105 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 4404019200 bytes (free: 4345167872, total: 25283526656).


  rollout step 200/500 | env_step=36000 grad_step=400 epsilon=0.976 | 9 tasks | mean task score 2.44/20
  rollout step 300/500 | env_step=39000 grad_step=400 epsilon=0.974 | 9 tasks | mean task score 1.11/20
  rollout step 400/500 | env_step=42000 grad_step=400 epsilon=0.972 | 16 tasks | mean task score 2.81/20
  rollout step 500/500 | env_step=45000 grad_step=400 epsilon=0.970 | 14 tasks | mean task score 2.64/20
env_step=45000 grad_step=500 | train loss=0.0075 q=0.179
env_step=45000 grad_step=600 | train loss=0.0154 q=0.418
  rollout step 100/500 | env_step=48000 grad_step=600 epsilon=0.968 | 13 tasks | mean task score 1.38/20
  rollout step 200/500 | env_step=51000 grad_step=600 epsilon=0.966 | 14 tasks | mean task score 2.07/20
  rollout step 300/500 | env_step=54000 grad_step=600 epsilon=0.964 | 12 tasks | mean task score 2.08/20
  rollout step 400/500 | env_step=57000 grad_step=600 epsilon=0.962 | 15 tasks | mean task score 2.33/20
  rollout step 500/500 | env_step=60000 grad_ste

## Push To The Hub

Run this cell to save the online-trained checkpoint. The inference notebook can load it later with `load_model`.


In [10]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")

Pushed to https://huggingface.co/micahr234/mouse-example-model-online
